<a href="https://colab.research.google.com/github/yasuhisasugiura-crypto/PARC2026_pre/blob/main/parc_subm_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================
# セル1：既存資産の確認
# ============================================
from pathlib import Path
import os

# 1) 学習済みモデルの存在確認
MERGED_MODEL_DIR = Path("/content/smolvla_libero_plus_spatial_lora_merged")
print(f"📦 学習済みモデル存在: {MERGED_MODEL_DIR.exists()}")

if MERGED_MODEL_DIR.exists():
    # 中身の確認
    for f in sorted(MERGED_MODEL_DIR.iterdir()):
        size_mb = f.stat().st_size / 1024 / 1024
        print(f"   {f.name}: {size_mb:.1f} MB")

# 2) 学習時のlerobotが残っているか確認
LEROBOT_DIR = Path("/content/lerobot/src/lerobot")
print(f"\n📦 lerobot存在: {LEROBOT_DIR.exists()}")

# 3) もし学習済みモデルが消えていたら Google Drive から復元
if not MERGED_MODEL_DIR.exists():
    print("\n⚠️ モデルが見つかりません")
    print("→ Google Drive に保存済みの merged.zip をアップロードしてください")

📦 学習済みモデル存在: False

📦 lerobot存在: False

⚠️ モデルが見つかりません
→ Google Drive に保存済みの merged.zip をアップロードしてください


In [2]:
# ============================================
# セル2：SmolVLAに必要な部分だけ抽出
# ============================================
import shutil
from pathlib import Path

# 元のlerobot
SRC_LEROBOT = Path("/content/lerobot/src/lerobot")

# 抽出先
DST_LEROBOT = Path("/content/submission/lerobot")

# 提出フォルダをクリーンに準備
SUBMISSION_DIR = Path("/content/submission")
if SUBMISSION_DIR.exists():
    shutil.rmtree(SUBMISSION_DIR)
SUBMISSION_DIR.mkdir(parents=True)

# 残すべきディレクトリのリスト
KEEP_DIRS = [
    "configs",           # 設定
    "policies/smolvla",  # SmolVLA本体
    "utils",             # ユーティリティ
    "optim",             # オプティマイザ
    "processor",         # 前処理・後処理
    "datasets",          # データセット関連（一部）
]

# 残すべき単独ファイル
KEEP_FILES = [
    "__init__.py",
    "constants.py",
    "errors.py",
    "policies/__init__.py",
    "policies/factory.py",
    "policies/normalize.py",
    "policies/pretrained.py",
]

# lerobotディレクトリを作成
DST_LEROBOT.mkdir(parents=True)

# ディレクトリを選択的にコピー
for keep_dir in KEEP_DIRS:
    src = SRC_LEROBOT / keep_dir
    dst = DST_LEROBOT / keep_dir
    if src.exists():
        shutil.copytree(src, dst)
        print(f"✅ コピー: {keep_dir}")
    else:
        print(f"⚠️ 存在せず: {keep_dir}")

# 単独ファイルをコピー
for keep_file in KEEP_FILES:
    src = SRC_LEROBOT / keep_file
    dst = DST_LEROBOT / keep_file
    if src.exists():
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print(f"✅ ファイルコピー: {keep_file}")

# サイズ確認
total_size = sum(f.stat().st_size for f in DST_LEROBOT.rglob("*") if f.is_file())
print(f"\n📊 lerobot抽出後サイズ: {total_size / 1024 / 1024:.1f} MB")
print(f"   （目標: 2〜3MB）")

⚠️ 存在せず: configs
⚠️ 存在せず: policies/smolvla
⚠️ 存在せず: utils
⚠️ 存在せず: optim
⚠️ 存在せず: processor
⚠️ 存在せず: datasets

📊 lerobot抽出後サイズ: 0.0 MB
   （目標: 2〜3MB）


In [3]:
# ============================================
# セル3：policies/__init__.py を軽量化
# ============================================

policies_init = DST_LEROBOT / "policies" / "__init__.py"

# SmolVLAだけをimportする最小版に書き換え
new_content = '''"""LeRobot policies (SmolVLA only for submission)"""

# SmolVLAだけをimport
from lerobot.policies.smolvla.configuration_smolvla import SmolVLAConfig
from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy

# BasePolicy相当の共通クラス
from lerobot.policies.pretrained import PreTrainedPolicy

__all__ = [
    "SmolVLAConfig",
    "SmolVLAPolicy",
    "PreTrainedPolicy",
]
'''

policies_init.write_text(new_content, encoding="utf-8")
print(f"✅ {policies_init} を軽量化しました")
print("   （act, diffusion, vqbet など他のimportを削除）")

FileNotFoundError: [Errno 2] No such file or directory: '/content/submission/lerobot/policies/__init__.py'

In [ ]:
# ============================================
# セル4：typing.Self を typing_extensions.Self に置換
# ============================================
import re

def patch_typing_self(directory):
    """
    ディレクトリ内の全.pyファイルで typing.Self を typing_extensions.Self に置換
    """
    patched_files = []
    total_replacements = 0

    for py_file in Path(directory).rglob("*.py"):
        content = py_file.read_text(encoding="utf-8")
        original = content

        # パターン1: from typing import ..., Self
        # 「Self」を除去し、typing_extensions から追加import
        pattern1 = r"from typing import ([^;\n]*?)Self([^;\n]*)"

        def replace_typing_self(m):
            before = m.group(1).rstrip(", ")
            after = m.group(2).lstrip(", ")
            # beforeとafterを整理
            typing_imports = []
            if before.strip():
                typing_imports.append(before.strip().rstrip(","))
            if after.strip():
                typing_imports.append(after.strip().rstrip(","))
            typing_str = ", ".join(t.strip(", ") for t in typing_imports if t.strip(", "))

            if typing_str:
                return f"from typing import {typing_str}\nfrom typing_extensions import Self"
            else:
                return "from typing_extensions import Self"

        content = re.sub(pattern1, replace_typing_self, content)

        if content != original:
            py_file.write_text(content, encoding="utf-8")
            patched_files.append(py_file)
            # 置換回数を数える
            n_replacements = original.count("Self") - content.count("Self") + \
                             content.count("from typing_extensions import Self")
            total_replacements += 1

    return patched_files, total_replacements

patched_files, total = patch_typing_self(DST_LEROBOT)
print(f"✅ 修正ファイル数: {len(patched_files)}")
for f in patched_files:
    print(f"   {f.relative_to(DST_LEROBOT)}")

# 修正後、typing.Self が残っていないか確認
import subprocess
result = subprocess.run(
    ["grep", "-R", "-l", "from typing import.*Self", str(DST_LEROBOT)],
    capture_output=True, text=True,
)
if result.stdout.strip():
    print(f"\n⚠️ まだ残っているファイル:\n{result.stdout}")
else:
    print("\n✅ typing.Self の直接import は全て修正済み")

In [4]:
# ============================================
# セル5：学習済みモデルをsubmissionへコピー
# ============================================

MODEL_WEIGHTS_DIR = SUBMISSION_DIR / "model_weights"
MODEL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)

# merged.zipの中身をコピー
for file_path in MERGED_MODEL_DIR.rglob("*"):
    if file_path.is_file():
        target = MODEL_WEIGHTS_DIR / file_path.relative_to(MERGED_MODEL_DIR)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(file_path, target)

# サイズ確認
weights_size = sum(f.stat().st_size for f in MODEL_WEIGHTS_DIR.rglob("*") if f.is_file())
print(f"✅ model_weights: {weights_size / 1024 / 1024:.1f} MB")

✅ model_weights: 0.0 MB


In [5]:
# ============================================
# セル6：policy_server.py を書き出し
# ============================================

policy_server_code = r'''"""
ポリシーサーバー v006 - 案A: 最小lerobot同梱版
"""
import argparse
import os
import sys
from abc import ABC, abstractmethod
from pathlib import Path

# 同梱lerobotをimportパスに追加
sys.path.insert(0, str(Path(__file__).parent))

import lerobot
import msgpack
import numpy as np
import torch
import uvicorn
from fastapi import FastAPI, Request, Response

from lerobot.policies.smolvla.modeling_smolvla import SmolVLAPolicy
from lerobot.configs import PreTrainedConfig


class BasePolicy(ABC):
    @abstractmethod
    def get_action(self, obs):
        pass
    @abstractmethod
    def reset(self, instruction=""):
        pass


class MyPolicy(BasePolicy):
    def __init__(self):
        weights_dir = Path(__file__).parent / "model_weights"
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        config = PreTrainedConfig.from_pretrained(weights_dir)
        config.device = self.device
        config.use_peft = False
        config.load_vlm_weights = True

        self.policy = SmolVLAPolicy.from_pretrained(
            weights_dir, config=config, strict=False
        )
        self.policy.to(self.device)
        self.policy.eval()
        self.instruction = ""

    def get_action(self, obs):
        with torch.no_grad():
            action_tensor = self.policy.select_action(obs)
            action = action_tensor.cpu().numpy().astype(np.float32).flatten()[:7]
        return action

    def reset(self, instruction=""):
        self.instruction = instruction
        if hasattr(self.policy, "reset"):
            self.policy.reset()


# サーバー部分（変更不可）
def deserialize_obs(data: bytes) -> dict:
    unpacked = msgpack.unpackb(data, raw=False)
    obs = {}
    for key, val in unpacked.items():
        arr = np.frombuffer(val["data"], dtype=np.dtype(val["dtype"]))
        obs[key] = arr.reshape(val["shape"]).copy()
    return obs

def serialize_action(action: np.ndarray) -> bytes:
    return msgpack.packb(
        {"data": action.astype(np.float32).tobytes()},
        use_bin_type=True,
    )

app = FastAPI(title="VLA Policy Server")
_policy = None

def set_policy(policy):
    global _policy
    _policy = policy

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/reset")
async def reset_policy(request: Request):
    body = await request.body()
    instruction = ""
    if body:
        import json
        data = json.loads(body)
        instruction = data.get("instruction", "")
    _policy.reset(instruction=instruction)
    return {"status": "ok"}

@app.post("/act")
async def act(request: Request):
    body = await request.body()
    obs = deserialize_obs(body)
    action = _policy.get_action(obs)
    return Response(
        content=serialize_action(action),
        media_type="application/x-msgpack",
    )

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--port", type=int, default=8000)
    parser.add_argument("--host", type=str, default="0.0.0.0")
    args = parser.parse_args()
    set_policy(MyPolicy())
    uvicorn.run(app, host=args.host, port=args.port)
'''

(SUBMISSION_DIR / "policy_server.py").write_text(policy_server_code, encoding="utf-8")
print(f"✅ policy_server.py 書き出し完了")

✅ policy_server.py 書き出し完了


In [6]:
# ============================================
# セル7：requirements.txt を書き出し
# ============================================

requirements = """# Policy server dependencies
fastapi>=0.68
uvicorn>=0.15
msgpack>=1.0
numpy>=1.19

# lerobot が使う軽量ライブラリ（成功者の書式を参考）
transformers>=4.30
safetensors
tokenizers
huggingface_hub
peft
draccus
num2words
einops
typing_extensions
"""

(SUBMISSION_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")
print(f"✅ requirements.txt 書き出し完了")

✅ requirements.txt 書き出し完了


In [8]:
# ============================================
# セル8：submission.zip 作成
# ============================================
import hashlib
from google.colab import files

SUBMISSION_ZIP = "/content/submission_v006.zip"

if os.path.exists(SUBMISSION_ZIP):
    os.remove(SUBMISSION_ZIP)

shutil.make_archive("/content/submission_v006", "zip", SUBMISSION_DIR)

# サイズ
size_mb = os.path.getsize(SUBMISSION_ZIP) / 1024 / 1024
print(f"📦 zip サイズ: {size_mb:.1f} MB")

# ハッシュ
sha256 = hashlib.sha256()
with open(SUBMISSION_ZIP, "rb") as f:
    for chunk in iter(lambda: f.read(8192), b""):
        sha256.update(chunk)
print(f"🔐 SHA256: {sha256.hexdigest()}")

# 中身の一覧
import zipfile
with zipfile.ZipFile(SUBMISSION_ZIP) as z:
    print("\n📋 zip中身（上位30件）:")
    for name in sorted(z.namelist())[:30]:
        print(f"   {name}")
    print(f"\n   合計 {len(z.namelist())} ファイル")
print(f"\n✅ 作成完了: {SUBMISSION_ZIP}")

📦 zip サイズ: 0.0 MB
🔐 SHA256: cd333253a250fd4275a79aba577ffe49e13e6f2e9e4c19c541f5b1d952ab4f73

📋 zip中身（上位30件）:
   lerobot/
   model_weights/
   policy_server.py
   requirements.txt

   合計 4 ファイル

✅ 作成完了: /content/submission_v006.zip


In [7]:
# ============================================
# セル9：Python 3.10 環境で validate_submission.py 実行
# ============================================
print("🔧 Step 1: Python 3.10 セットアップ...")
!apt-get install -y python3.10 python3.10-venv python3.10-dev 2>&1 | tail -3

print("\n🔧 Step 2: venv 作成...")
!rm -rf /tmp/py310_venv
!python3.10 -m venv /tmp/py310_venv

print("\n🔧 Step 3: 検証ツールインストール...")
!/tmp/py310_venv/bin/pip install --upgrade pip -q
!/tmp/py310_venv/bin/pip install -q msgpack numpy requests fastapi uvicorn packaging psutil

print("\n🔧 Step 4: validate_submission.py 取得...")
!wget -q https://raw.githubusercontent.com/matsuolab/PARC2026_pre/main/validate_submission.py \
    -O /tmp/validate_submission.py

print("\n" + "="*60)
print("📋 Step 5A: 静的チェック（--static、30秒）")
print("="*60)
!/tmp/py310_venv/bin/python /tmp/validate_submission.py {SUBMISSION_ZIP} --static

print("\n" + "="*60)
print("🚀 Step 5B: 完全検証（--install、5〜15分）")
print("="*60)
print("⚠️ 依存インストールとサーバー起動に時間がかかります")
!/tmp/py310_venv/bin/python /tmp/validate_submission.py {SUBMISSION_ZIP} --install

🔧 Step 1: Python 3.10 セットアップ...
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/main/p/python3.10/python3.10-dev_3.10.12-1%7e22.04.15_amd64.deb  404  Not Found [IP: 185.125.190.83 80]
E: Failed to fetch http://security.ubuntu.com/ubuntu/pool/universe/p/python3.10/python3.10-venv_3.10.12-1%7e22.04.15_amd64.deb  404  Not Found [IP: 185.125.190.83 80]
E: Unable to fetch some archives, maybe run apt-get update or try with --fix-missing?

🔧 Step 2: venv 作成...
The virtual environment was not created successfully because ensurepip is not
available.  On Debian/Ubuntu systems, you need to install the python3-venv
package using the following command.

    apt install python3.10-venv

You may need to use sudo with that command.  After installing the python3-venv
package, recreate your virtual environment.

Failing command: /tmp/py310_venv/bin/python3.10


🔧 Step 3: 検証ツールインストール...
/bin/bash: line 1: /tmp/py310_venv/bin/pip: No such file or directory
/bin/bash: line 1: /tmp/py310_venv/bin